In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

In [6]:
df = pd.read_csv("data/all_human_rps.csv")

In [7]:
# 원핫 인코딩 (0=가위, 1=바위, 2=보)
encoder = OneHotEncoder(categories=[[0,1,2]], sparse_output=False)
moves_encoded = encoder.fit_transform(df[['move']])

# 시퀀스 데이터 준비 (최근 10라운드 입력 → 다음 행동 예측)
X_seq, y_seq = [], []
for i in range(len(moves_encoded) - SEQ_LEN):
    seq = moves_encoded[i:i+SEQ_LEN]
    target = df['move'].iloc[i+SEQ_LEN]
    X_seq.append(seq)
    y_seq.append(target)

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

print(f"입력 시퀀스 형태: {X_seq.shape}, 타겟 형태: {y_seq.shape}")

# 모델 정의
model = Sequential([
    Input(shape=(SEQ_LEN, 3)),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

model.compile(optimizer=Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 모델 학습
model.fit(X_seq, y_seq, validation_split=0.2, epochs=10, batch_size=64, verbose=1)

입력 시퀀스 형태: (24990, 10, 3), 타겟 형태: (24990,)
Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.3828 - loss: 1.0919 - val_accuracy: 0.4538 - val_loss: 1.0696
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.4489 - loss: 1.0655 - val_accuracy: 0.4742 - val_loss: 1.0530
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.4654 - loss: 1.0491 - val_accuracy: 0.4746 - val_loss: 1.0420
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.4711 - loss: 1.0422 - val_accuracy: 0.4802 - val_loss: 1.0396
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.4878 - loss: 1.0286 - val_accuracy: 0.4808 - val_loss: 1.0356
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.4837 - loss: 1.0309 - val_accuracy: 0.4884 - val_loss: 1.0315
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.4833 - loss: 1.0290 - val_accuracy: 0.4830 - val_loss: 1.0322
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accurac

In [8]:
# 모델 저장
model.save("models/rps_ai_base.keras")
print("모델 저장 완료: rps_ai_base.keras")

모델 저장 완료: rps_ai_base.keras
